In [ ]:
import Pkg;
Pkg.activate("../")
Pkg.update()
Pkg.instantiate()
#Pkg.add(url = "https://github.com/RalphAS/Pseudospectra.jl")

In [ ]:
using Pseudospectra, Plots

In [ ]:
Pkg.status()

# Our example

We start by defining the dynamic, a perturbation of the doubling map
$$
T(z) = i z^2 \cdot exp\left[\left(\frac{1}{2}-\frac{\pi}{16}\right)\left(z-\frac{1}{z}\right)\right].
$$

In [ ]:
T(z) = im*z^2*exp((1/2-π/16)*(z-1/z))

# Non rigorous exploration of the Pseudospectrum

We fix the truncation size for the Galerkin approximation.

In [ ]:
K = 128
N = 2*K+1

We start by writing the Blashke product as an interval map

In [ ]:
S(x) = 2 * x + (1 / (2 * pi) - 1 / 16) * sin(2 * π * x) + 0.25

In [ ]:
using Plots
plot(S, 0, 1)

In [ ]:
using RigorousInvariantMeasures
FourierBasis = RigorousInvariantMeasures.FourierAdjoint(K, 65536)
P = DiscretizedOperator(FourierBasis, S)

In [ ]:
import IntervalArithmetic
midI = IntervalArithmetic.mid
radI = IntervalArithmetic.radius

midP = midI.(real.(P.L)) + im * midI.(imag.(P.L))
spectralportrait(midP)

# Numerical oracle for the constants

In this section we do some numerical computations to narrow down the set of parameters. Later on, we will use self-validated numerical methods (Interval Arithmetic) to certify the 
numerical values we computed now. We do this since numerical computations are inexpensive, while self validated methods may be more time-consuming.

In [ ]:
using Plots

In [ ]:
Pkg.status()

Let $A_{r} = \{z \mid e^{-2\pi r}\leq |z| \leq e^{2\pi r}\}$.

We are interested in finding $\eta$, $\rho$ such that the closure $A_{\rho}$ is contained in $B_{\mu}(A_{\eta})$.
We are interested in maximizing $\alpha-\eta$, since it is the constant appearing in the main error term of our functional analytic treatment, i.e.:
$$
||Lf-L_Kf||_{\mathcal{A}_0}\leq \left(1+\frac{2}{e^{2 \pi (\rho-\alpha)}-1}\right)\left(e^{-2\pi K\alpha}+e^{-2\pi K(\alpha-\eta)}\right)||f||_{\mathcal{A}_{\alpha}}.	
$$

For $\eta>1$ fix
$$
\rho_o(\eta):=\frac{1}{2\pi}\log\left(\min_{\theta \in [0,1]}|B_{\mu}(e^{2\pi\eta} e^{2\pi i \theta})|\right)
$$
where $_0$ stays for outer.
We would like to maximize this function.

In [ ]:
ρ_o(η) = log(minimum(abs.(T.([exp(2 * π*(η + im * θ)) for θ in 0:0.001:1]))))/(2*π)

In [ ]:
plot(η->ρ_o(η)-η, 0, 0.5)

Similarly, we would like to treat the image inside the circle; for $\eta>1$ we define 
$$
\rho_i(\eta) :=-\frac{1}{2\pi}\log\left(\max_{\theta \in [0, 1]}\left|B_{\mu}(e^{2\pi (-\eta + i \theta)})\right|\right)
$$

In [ ]:
ρ_i(η) = -log(maximum(abs.(T.([exp(2 * π*(-η + im * θ)) for θ in 0:0.001:1]))))/(2*π)

In [ ]:
plot(η->ρ_i(η)-η, 0, 0.5)

We define now 
$$
\rho(\eta) = \min\{\rho_i(\eta),\rho_o(\eta)\}-\eta
$$
we have that our dynamic is expanding the annulus when this function is positive.

In [ ]:
ρ(η) = min(ρ_i(η), ρ_o(η))

We plot now the function 
$$
    \eta \mapsto \rho(\eta)-\eta
$$
our dynamic is expanding the annulus when this function is positive.

In [ ]:
plot(η -> (ρ(η)-η), 0.15, 0.2)

From this non certified plot, we can see that the the difference $\rho(\eta)-\eta$ seems to have a maximum at  
$0.0460$.

We need now to be careful in choosing $\eta$ and $\rho$ in such a way that our computation is well behaved.

By our numerical exploration of the pseudospectrum, we want to isolate the eigenvalues outside of a circle of radius $0.5$.

By Lemma 3.10 and Lemma 3.13, we have that 
$$
\frac{||f||_{\mathcal{A}_{\alpha}}}{||f||_{\mathcal{A}_{0}}}\leq (1/\mu)^{\frac{\alpha}{\alpha-\eta}} \left(1+\frac{2}{e^{2 \pi (\rho - \alpha)} - 1}\right)^{\frac{\alpha}{\alpha-\eta}} 
$$
and
$$
||\mathcal{L}-\mathcal{L}_K||_{\mathcal{A}_{\alpha}\to \mathcal{A}_0}\leq \left(1+\frac{2}{e^{2 \pi (\rho - \alpha)} - 1}\right)\left(e^{-2\pi K\alpha}+e^{-2\pi K(\alpha-\eta)}\right).
$$

We refer to Proposition 2 in the paper, what we would like to control and make small is
$$
||\mathcal{L}-\mathcal{L}_K||_{\mathcal{A}_{\alpha}\to \mathcal{A}_0}\frac{||f||_{\mathcal{A}_{\alpha}}}{||f||_{\mathcal{A}_{0}}};
$$
since $\rho>\alpha$ we have then that $\alpha-\eta$ is at most $0.046$.

To optimize this, we pass to the logarithm.

In [ ]:
bound(η, ρ, α; K, μ) =  α/(α-η)*log2(1/μ)+(α/(α-η)+1)*log2(1+2/(exp(2*π*(ρ-α))-1))+log2(exp(-2*π*K*α)+exp(-2*π*K*(α-η)))

In [ ]:
bound_η_α(η, α; K, μ) = bound(η, ρ(η), α; K, μ)

In [ ]:
function bound_η(η; K, μ)
    η_eps = η+(ρ(η)-η)/100
    ρ_eps = ρ(η)-(ρ(η)-η)/100
    val, index = findmin([bound_η_α(η, α; K, μ) for α in LinRange(η_eps, ρ_eps, 100)])
    return val, LinRange(η_eps, ρ_eps, 100)[index]
end

In [ ]:
xmin = 0.1
xmax = 0.28
μ = 0.26
plot(η -> bound_η(η; K = 256, μ)[1], xmin, xmax, label = "256")
plot!(η -> bound_η(η; K = 128, μ)[1], xmin, xmax, label = "128")
plot!(η -> bound_η(η; K = 64, μ)[1], xmin, xmax, label = "64")
plot!(η -> bound_η(η; K = 512, μ)[1], xmin, xmax, label = "512")
plot!(η -> bound_η(η; K = 1024, μ)[1], xmin, xmax, label = "1024")
plot!(η -> 0, label = "0")

We choose a discretization size of $K=256$. For this discretization size we fix $\eta = 0.45175$

In [ ]:
steps = 200
val, index = findmin([bound_η(η; K = 256, μ)[1] for η in LinRange(0.1, 0.28, steps)])

In [ ]:
chosen_η = LinRange(0.1, 0.28, steps)[index]
chosen_α = bound_η(chosen_η; K = 256, μ)[2]

By Proposition 3.14, we have that the eigenvalues of modulus bigger than $1/2$ of $\mathcal{L}$ are contained 
in $\sigma_\delta$ for all $\delta\geq 2^{-65}$, where 
$$
\sigma_{\delta} = \sigma(\mathcal{L_K})\cup \{z \in \mathbb{C} \mid z-\mathcal{L}_K\textrm{ is bounded invertible and } |(z-\mathcal{L}_K)^{-1}|> \delta^{-1}|\}
$$

# Certifying the constants

For the specific values of $\alpha$, $\rho$ and $\eta$ computed above, we will certify the value of the constants.

We will repeat the process above to estimate $\rho$, by enclosing the image of the annulus of radius $2\pi\eta$, by using now interval arithmetic to 
obtain guaranteed enclosures.

In [ ]:
N = 32768
IΘ = [Interval(i, i+1)/N for i in 0:N-1];

Interval arithmetics....

In [ ]:
ρ_o_c(η; IΘ) = log(minimum(abs.(T.([exp(2 * π*(η + im * θ)) for θ in IΘ]))))/(2*π)
ρ_i_c(η; IΘ) = -log(maximum(abs.(T.([exp(2 * π*(-η + im * θ)) for θ in IΘ]))))/(2*π)
ρ_c(η; IΘ) = min(ρ_i_c(η; IΘ), ρ_o_c(η; IΘ))

For the chosen $\eta$ we have that our oracle for $\rho$ returns

In [ ]:
ρ(chosen_η)

The certificate $\rho$ has some overestimates, but we can make this error small by taking a big partition $I\Theta$

In [ ]:
ρ_c(chosen_η; IΘ)

In [ ]:
ρ_certified = Interval((ρ_c(chosen_η; IΘ)).lo) 

After all computations, our chosen $\alpha$, $\eta$ and certified $\rho$ are

In [ ]:
chosen_η, ρ_certified, chosen_α

And the certified bound is 

In [ ]:
δ = 2^bound(Interval(chosen_η), ρ_certified, Interval(chosen_α); K = 256, μ)

# Establishing the enclosing curves

In [ ]:
K = 256
FourierBasis = RigorousInvariantMeasures.FourierAdjoint(K, 65536)
P = DiscretizedOperator(FourierBasis, S)
midP = midI.(real.(P.L)) + im * midI.(imag.(P.L))

In [ ]:
using LinearAlgebra
F = schur(midP)
eigs = diag(F.T)

The following are the eigenvalues of the Schur decomposition we are using to enclose the spectrum of the abstract operator.

In [ ]:
λ_1 = 1.0
λ_2 = eigs[1]
λ_3 = eigs[2]

We check that the circle of radius $0.26$ effectively cuts off the reset of the eigenvalues of the Schur descomposition.

In [ ]:
abs.(eigs)

We are going to enclose the eigenvalues by four circles:
* the circle $C_{\gamma_0}$ centered in $0$ with radius $0.26$
* the circle $C_{\gamma_1}$ centered at $\lambda_1$ with radius $0.01$
* the circle $C_{\gamma_2}$ centered at $\lambda_2$ with radius $0.01$
* the circle $C_{\gamma_3}$ centered at $1$ with radius $0.1$

This computation is quite expensive, but we ran an example of it in this notebook to show how it works.
In the directory *ExperimentsPseudospectra.jl/scripts/Arnold* the scripts that run these computations using parallelization can be found,
for local computation and computation on a slurm cluster.

This is the typical output of the computation, here certifying the eigenvalue at $1$

```┌ Info: Added 8 processes
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/Arnold/local/setup.jl:17
AMD Ryzen 5 5600 6-Core Processor: 
          speed         user         nice          sys         idle          irq
#1-12  4441 MHz     134692 s        120 s     121729 s     627092 s          0 s
┌ Info: Switching to OpenBLAS with ConsistentFPCSR = 1 flag enabled, guarantees
│         correct floating point rounding mode over all threads.
└ @ BallArithmetic /home/isaia/.julia/packages/BallArithmetic/nPayJ/src/BallArithmetic.jl:20
┌ Info: OpenBLAS is giving correct rounding on a (1024,1024) test matrix on 6 threads
└ @ BallArithmetic /home/isaia/.julia/packages/BallArithmetic/nPayJ/src/BallArithmetic.jl:27
┌ Info: Schur decomposition errors
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/Arnold/local/setup.jl:33
┌ Info: ("E_M", 4.362697431116958e-13)
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/Arnold/local/setup.jl:38
┌ Info: ("E_T", 5.447296963610192e-9)
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/Arnold/local/setup.jl:39
┌ Info: ("norm_Z", Ball{Float64, Float64}(1.000000000000006, 2.336908444533492e-12))
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/Arnold/local/setup.jl:40
┌ Info: ("norm_Z_inv", Ball{Float64, Float64}(1.0000000000000058, 2.3373525337433425e-12))
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/Arnold/local/setup.jl:41
┌ Info: ("Certifying ", 1.0, "radius", 0.1, "radius pearl", 0.049000000000000016)
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/Arnold/local/run_Arnold_one.jl:11
┌ Info: 13 svd need to be computed to certify the arc centered at 1.0, with radius 0.1, with pearls of size 0.049000000000000016
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/script_functions.jl:37
┌ Info: with start angle 0 and stop angle 6.283185307179586
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/script_functions.jl:38
┌ Info: Jobs submitted to the queue
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/script_functions.jl:42
┌ Info: The minimum singular value along the arc centered at 1.0, with radius 0.1, with pearls of size 0.049000000000000016 with start angle 0 and stop angle 6.283185307179586 is 0.050388243829828226, the maximum of the l2 pseudospectra is bounded by 19.84589904298336
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/script_functions.jl:81
┌ Info: ("Average time for certifying an SVD", 10.905107298384618)
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/script_functions.jl:83
┌ Info: ("Total time for certifying the arc", 20.453904849)
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/script_functions.jl:84
┌ Info: ("The norm of the resolvent for the discretized operator is", 898.9997063371694)
└ @ Main /home/isaia/Code/ExperimentsPseudospectra.jl/scripts/Arnold/local/cleaning.jl:10```

In [ ]:
F = schur(midP)

In [ ]:
circ = [0.26*(cos(2*pi*θ)+im*sin(2*pi*θ)) for θ in 0.0:0.01:1.0]

In [ ]:
num_val = [1/svd(z*I-F.T).S[end] for z in circ]

In [ ]:
plot(num_val)

In [ ]:
function vals(T, circ)
    num_val = zeros(length(circ), 3)
    for (i, z) in enumerate(circ) 
        K = svd(z*I-F.T)
        num_val[i, :] = [K.S[end]; K.S[end-1]; K.S[end-1]-K.S[end]]
    end
    return num_val
end

In [ ]:
v = vals(T, circ)

In [ ]:
plot(v[:, 1])

In [ ]:
plot(v[:, 2])

In [ ]:
minimum(v[:, 2])

In [ ]:
function prepare_greedy(c, r, Nstart, η, T)
    initial_arc_list = []
    for i in 1:Nstart
        z_i = r*(cos((2*pi*i)/Nstart)+im*sin((2*pi*i)/Nstart))
        z_i_plus = r*(cos((2*pi*(i+1))/Nstart)+im*sin((2*pi*(i+1))/Nstart))
        push!(initial_arc_list, (z_i, z_i_plus))
    end
    queue = initial_arc_list
    result = []
    while !isempty(queue)
        (z_a, z_b) = pop!(queue)
        l = abs(z_a-z_b)
        K = svd(z_a*I-T)
        σ_n = K.S[end]
        #@info l, σ_n
        if l/(σ_n) < η
            push!(result, (z_a, z_b))
        else
            z_m = (z_a + z_b)/2
            push!(queue, (z_a, z_m))
            push!(queue, (z_m, z_b))
        end
        K = length(result)
        if K!=0 && K % 1000 == 0
            @info K, length(queue)
        end
    end    
    return result
end

In [ ]:
w = prepare_greedy(λ_2, 0.01, 1024, 0.1, F.T)